In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: QedmaによるQiskit Function
> **Note:** Qiskit Functionsは、IBM Quantum&reg; Premiumプラン、Flexプラン、およびOn-Prem（IBM Quantum Platform API経由）プランのユーザーのみが利用できる実験的な機能です。プレビューリリースの状態であり、変更される場合があります。

## 概要
近年、量子処理ユニット（QPU）は大幅に改善されていますが、既存のハードウェアにおけるノイズや不完全性によるエラーは、量子アルゴリズム開発者にとって依然として中心的な課題です。この分野が古典的に検証できないユーティリティスケールの量子計算に近づくにつれて、精度を保証しながらノイズをキャンセルするソリューションがますます重要になっています。この課題を克服するため、QedmaはQuantum Error Suppression and Error Mitigation（QESEM）を開発しました。これは[Qiskit Function](/guides/functions)として、IBM Quantum Platformにシームレスに統合されています。

QESEMを使用することで、ユーザーはノイズの多いQPU上で量子回路を実行し、基本的な限界に近い非常に効率的なQPU時間オーバーヘッドで、高精度なエラーのない結果を得ることができます。これを実現するために、QESEMはQedmaが開発した独自の手法のスイートを活用して、エラーの特性評価と低減を行っています。エラー低減技術には、ゲート最適化、ノイズを考慮したトランスパイレーション、エラー抑制（ES）、および不偏エラー緩和（EM）が含まれます。これらの特性評価に基づく手法を組み合わせることで、ユーザーは汎用的な大容量量子回路に対して信頼性の高いエラーのない結果を達成でき、それ以外では実現できないアプリケーションを解き放ちます。

基礎となるコンポーネントの詳細な説明とユーティリティスケールのデモについては、論文[Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)を参照してください。
## 説明
QedmaのQESEM functionを使用することで、エラー抑制と緩和を適用した回路の見積もりと実行を簡単に行い、より大きな回路容量とより高い精度を達成できます。QESEMを使用するには、量子回路、測定するオブザーバブルのセット、各オブザーバブルの目標統計精度、および選択したQPUを指定します。目標精度まで回路を実行する前に、回路の実行を必要としない解析的計算に基づいて必要なQPU時間を見積もることができます。QPU時間の見積もりに納得したら、QESEMで回路を実行できます。

回路を実行すると、QESEMはお使いの回路に合わせたデバイス特性評価プロトコルを実行し、回路内で発生するエラーの信頼性の高いノイズモデルを生成します。この特性評価に基づいて、QESEMはまずノイズを考慮したトランスパイレーションを実装し、入力回路を物理量子ビットとゲートのセットにマッピングして、目標オブザーバブルに影響するノイズを最小化します。これには、ネイティブで使用可能なゲート（IBM&reg;デバイスのCX/CZ）と、QESEMによって最適化された追加のゲートが含まれ、QESEMの拡張ゲートセットを形成しています。次に、QESEMはQPU上で特性評価に基づくESおよびEM回路のセットを実行し、その測定結果を収集します。これらは古典的な後処理によって、要求された精度に対応する各オブザーバブルの不偏期待値とエラーバーを提供します。

![Qedma QESEM overview](../docs/images/guides/qedma-qesem/overview.svg)
QESEMは、さまざまな量子アプリケーションに対して高精度な結果を提供し、現在達成可能な最大の回路容量での実証がなされています。QESEMは以下のユーザー向け機能を提供しており、以下のベンチマークセクションで実証されています：
-	**精度の保証:** QESEMはオブザーバブルの期待値に対する不偏推定を出力します。そのEM手法は理論的な保証を備えており、Qedmaの最先端の特性評価と組み合わせることで、緩和がユーザー指定の精度までノイズのない回路出力に収束することを保証します。系統的なエラーや偏りが生じやすい多くのヒューリスティックなEM手法とは対照的に、QESEMの精度保証は、汎用的な量子回路とオブザーバブルにおいて信頼性の高い結果を確保するために不可欠です。
-	**大規模QPUへのスケーラビリティ:** QESEMのQPU時間は回路容量に依存しますが、それ以外は量子ビット数に依存しません。Qedmaは、IBM Quantum 127量子ビットEagleおよび133量子ビットHeronデバイスを含む、現在利用可能な最大の量子デバイスでQESEMを実証しています。
-	**アプリケーション非依存:** QESEMは、ハミルトニアンシミュレーション、VQE、QAOA、振幅推定など、さまざまなアプリケーションで実証されています。ユーザーは任意の量子回路と測定するオブザーバブルを入力し、正確なエラーのない結果を得ることができます。唯一の制限は、アクセス可能な回路容量と出力精度を決定するハードウェア仕様と割り当てられたQPU時間によって決まります。対照的に、多くのエラー低減ソリューションはアプリケーション固有であるか、制御されていないヒューリスティックを含んでおり、汎用的な量子回路とアプリケーションには適用できません。
-  **拡張ゲートセット:** QESEMは分数角度ゲートをサポートし、IBM Quantum EagleデバイスでQedma最適化の分数角度$Rzz(\theta)$ゲートを提供します。この拡張ゲートセットにより、より効率的なコンパイルが可能になり、デフォルトのCX/CZコンパイルと比較して最大2倍の回路容量が利用可能になります。
-	**マルチベースオブザーバブル:** QESEMは、汎用的なハミルトニアンなど、多くの非可換Pauli文字列で構成された入力オブザーバブルをサポートします。測定ベースの選択とQPUリソース配分（ショットと回路）の最適化は、要求された精度に必要なQPU時間を最小化するためにQESEMによって自動的に実行されます。ハードウェアの忠実度と実行レートを考慮したこの最適化により、より深い回路を実行してより高い精度を得ることができます。
## ベンチマーク
QESEMは、さまざまなユースケースとアプリケーションでテストされています。以下の例は、QESEMで実行できるワークロードの種類を評価するのに役立ちます。

所与の回路とオブザーバブルに対して、エラー緩和と古典的シミュレーションの両方の難しさを定量化するための主要な評価指標は**アクティブボリューム**です：回路内でオブザーバブルに影響するCNOTゲートの数です。アクティブボリュームは、回路の深さと幅、オブザーバブルの重み、およびオブザーバブルのライトコーンを決定する回路構造に依存します。詳細については、[2024 IBM Quantum Summit](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s)のトークを参照してください。QESEMは高ボリュームの領域で特に大きな価値を提供し、汎用的な回路とオブザーバブルに対して信頼性の高い結果をもたらします。

![Active volume](../docs/images/guides/qedma-qesem/active_volume.svg)

| アプリケーション    | 量子ビット数 | デバイス | 回路の説明 | 精度 | 合計時間 | ランタイム使用量 |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE回路  | 8              | Eagle (r3) | 21層合計、9測定ベース、1Dチェーン                    | 98%      | 35分      | 14分         |
| Kicked Ising   | 28              | Eagle (r3) | 3ユニーク層 × 3ステップ、2D heavy-hexトポロジー                      | 97%     | 22分      | 4分          |
| Kicked Ising   | 28              | Eagle (r3) | 3ユニーク層 × 8ステップ、2D heavy-hexトポロジー                     | 97%      | 116分      | 23分          |
| トロッタライズドハミルトニアンシミュレーション   | 40  | Eagle (r3)            | 2ユニーク層 × 10トロッターステップ、1Dチェーン                    | 97%      | 3時間     | 25分         |
| トロッタライズドハミルトニアンシミュレーション   | 119  | Eagle (r3)           | 3ユニーク層 × 9トロッターステップ、2D heavy-hexトポロジー                    | 95%      | 6.5時間     | 45分         |
| Kicked Ising  | 136             | Heron (r2) | 3ユニーク層 × 15ステップ、2D heavy-hexトポロジー                    | 99%      | 52分             | 9分           |

ここでの精度はオブザーバブルの理想値に対して相対的に測定されています：$\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$。ここで「$\epsilon$」は緩和の絶対精度（ユーザー入力で設定）であり、$\langle O \rangle_{ideal}$はノイズのない回路でのオブザーバブルです。
「ランタイム使用量」はバッチモードでのベンチマークの使用量を測定します（個々のジョブの使用量の合計）。一方、「合計時間」はセッションモードでの使用量を測定します（実験のウォール時間）。これには追加の古典的処理と通信時間が含まれます。QESEMは両方のモードでの実行が可能であり、ユーザーは利用可能なリソースを最大限に活用できます。

28量子ビットのKicked Ising回路は、ibm_kawasakiの3つの連結ループ上で、Shinjo et al.が研究した離散時間準結晶（[arXiv 2403.16718](https://arxiv.org/abs/2403.16718)および[Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)を参照）をシミュレートします。ここで使用した回路パラメータは$(\theta_x, \theta_z) = (0.9 \pi, 0)$で、強磁性初期状態$| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$を用いています。測定されたオブザーバブルは磁化の絶対値$M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$です。ユーティリティスケールのKicked Ising実験はibm_fezの最良の136量子ビット上で実行されました。この特定のベンチマークはClifford角$(\theta_x, \theta_z) = (\pi, 0)$で実行されました。この角度では、アクティブボリュームが回路深さとともにゆっくりと増加するため、高いデバイス忠実度と合わせて、短いランタイムで高精度が実現できます。

トロッタライズドハミルトニアンシミュレーション回路は、分数角度での横断磁場Isingモデル用のものです：$(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$および$(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$（[Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)を参照）。ユーティリティスケールの回路はibm_brisbaneの最良の119量子ビット上で実行され、40量子ビットの実験は利用可能な最良のチェーン上で実行されました。精度は磁化について報告されており、より高い重みのオブザーバブルについても高精度な結果が得られています。

VQE回路は、Deutsches Elektronen-Synchrotron（DESY）の量子技術・応用センターの研究者と共同で開発されました。ここでの目標オブザーバブルは、多数の非可換Pauli文字列からなるハミルトニアンであり、マルチベースオブザーバブルに対するQESEMの最適化されたパフォーマンスを強調しています。緩和は古典的に最適化されたansatzに適用されました。これらの結果はまだ未発表ですが、同様の構造的特性を持つ異なる回路についても同等の品質の結果が得られます。
## はじめに
[IBM Quantum Platform APIキー](http://quantum.cloud.ibm.com/)を使用して認証し、以下のようにQESEM Qiskit Functionを選択します。（このスニペットは、すでに[アカウントを保存済み](/guides/functions#install-qiskit-functions-catalog-client)であることを前提としています。）

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## 例
はじめに、所与の`pub`についてQESEMを実行するために必要なQPU時間を見積もる基本的な例を試してみましょう：

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

以下の例はQESEMジョブを実行します：

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

使い慣れたQiskit Serverless APIを使用して、Qiskit Functionワークロードのステータスを確認したり、結果を返したりすることができます：

In [ ]:
print(sample_job.status())
result = sample_job.result()

## 関数パラメータ
| 名前 |  型 | 説明 | 必須 | デフォルト |  例 |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) |これはメインの入力です。`Pub`には2〜4つの要素が含まれます：回路、1つ以上のオブザーバブル、0または単一のパラメータ値セット、およびオプションの精度。精度が指定されていない場合は、`options`の`default_precision`が使用されます|  はい|  N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string|使用するバックエンドの名前 |いいえ | QESEMはIBMが報告する最も空いているデバイスを取得します| `"ibm_fez"`|
| `instance` | string|  その形式で使用するインスタンスのクラウドリソース名 |  いいえ |  N/A | `"CRN"`  |
| `options` | dictionary |入力オプション。詳細については**Options**セクションを参照してください。 | いいえ |  詳細については**Options**セクションを参照してください。    |  `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}`  |

### オプション
| オプション |  選択肢 | 説明 | デフォルト |
| -----| -----------| -------- | ------- |
| `estimate_time_only` |  `"analytical"`  / `"empirical"` / None  | 設定すると、QESEMジョブはQPU時間の見積もりのみを計算します。詳細については以下の説明を参照してください。Noneに設定すると、回路はQESEMで実行されます| None |
|`default_precision` | 0 < float | 精度が設定されていない`pubs`に適用されます。精度は絶対値でのオブザーバブルの期待値に対する許容誤差を示します。つまり、緩和のためのQPUランタイムは、対象のすべてのオブザーバブルの出力値が目標精度の`1σ`信頼区間内に収まるように決定されます。複数のオブザーバブルが指定された場合、緩和は各入力オブザーバブルについて目標精度に達するまで実行されます。 | 0.02|
|`max_execution_time` | 0 < integer < 28,800（8時間）| QESEMプロセス全体に使用するQPU時間を秒単位で制限できます。詳細については以下を参照してください。| 3,600（1時間）|
| `transpilation_level` | 0 / 1 | 以下の説明を参照してください | 1|
| `execution_mode` | `"session"` / `"batch"` | 以下の説明を参照してください | "batch"|

> **Caution:** QPU時間の見積もりはバックエンドによって異なります。そのため、QESEM functionを実行する際は、QPU時間の見積もりを取得した際に選択したものと同じバックエンドで実行することを確認してください。

> **Note:** QESEMは、目標精度に達するか`max_execution_time`に達するか、どちらか先に来た方で実行を終了します。

- `estimate_time_only` - このフラグを使用すると、QESEMで回路を実行するために必要なQPU時間の見積もりを取得できます。
    - `"analytical"`に設定すると、QPU使用量を消費せずにQPU時間の上限が計算されます。この見積もりは30分の解像度（例：30分、60分、90分など）を持っています。通常は悲観的であり、単一のPauliオブザーバブルまたはサポートが交差しないPauliの和（例：Z0+Z1）についてのみ取得できます。これは主に、ユーザーが提供するさまざまなパラメータ（回路、精度など）の複雑さレベルを比較するのに役立ちます。
    - より正確なQPU時間の見積もりを得るには、このフラグを`"empirical"`に設定してください。このオプションでは少数の回路を実行する必要がありますが、大幅に正確なQPU時間の見積もりが得られます。この見積もりは5分の解像度（例：20分、25分、30分など）を持っています。ユーザーはバッチモードまたはセッションモードのいずれかで経験的時間見積もりを実行することを選択できます。詳細については、`execution_mode`の説明を参照してください。例えば、バッチモードでは、経験的時間見積もりは10分未満のQPU時間を消費します。

- `max_execution_time`: QESEMプロセス全体に使用するQPU時間を秒単位で制限できます。目標精度に達するために必要な最終的なQPU時間はQESEMジョブ中に動的に決定されるため、このパラメータにより実験のコストを制限できます。動的に決定されたQPU時間がユーザーが割り当てた時間よりも短い場合、このパラメータは実験に影響しません。`max_execution_time`パラメータは、ジョブ開始前にQESEMが提供する解析的な時間見積もりが悲観的すぎ、それでも緩和ジョブを開始したい場合に特に役立ちます。時間制限に達すると、QESEMは新しい回路の送信を停止します。すでに送信された回路は引き続き実行されます（そのため合計時間が制限を最大30分超える場合があります）。ユーザーはその時点まで実行された回路の処理済み結果を受け取ります。解析的な時間見積もりよりも短いQPU時間制限を適用したい場合は、Qedmaに相談して、時間制限内で達成可能な精度の見積もりを取得してください。

- `transpilation_level`: 回路がQESEMに送信されると、自動的にいくつかの代替回路トランスパイレーションを準備し、QPU時間を最小化するものを選択します。例えば、代替トランスパイレーションでは、Qedma最適化の分数RZZゲートを使用して回路深さを削減する場合があります。もちろん、すべてのトランスパイレーションは理想的な出力の観点から入力回路と等価です。回路のトランスパイレーションをより細かく制御するには、`options`でトランスパイレーションレベルを設定してください。`"transpilation_level": 1`が上記のデフォルト動作に対応する一方で、`"transpilation_level": 0`は元の回路に必要な最小限の修正のみを含みます。例えば、「レイヤー化」（回路操作を同時の2量子ビットゲートの「レイヤー」に整理すること）などです。いずれの場合も、高忠実度量子ビットへの自動ハードウェアマッピングが適用されることに注意してください。

| transpilation_level | 説明 |
|:-:|:--|
| `1` | デフォルトのQESEMトランスパイレーション。いくつかの代替トランスパイレーションを準備し、QPU時間を最小化するものを選択します。バリアはレイヤー化ステップで修正される場合があります。 |
| `0` | 最小限のトランスパイレーション：緩和された回路は構造的に入力回路に近いものになります。レベル0で提供される回路はデバイスの接続性に合致しており、以下のゲートで指定される必要があります：CX、Rzz(α)、および標準的な単一量子ビットゲート（U、x、sx、rzなど）。バリアはレイヤー化ステップで尊重されます。 |

- `execution_mode` - ユーザーは、専用の[IBMセッション](/guides/execution-modes#session-mode)または複数の[IBMバッチ](/guides/execution-modes#batch-mode)でQESEMジョブを実行することを選択できます：
    -   **セッションモード**: セッションはよりコストがかかりますが、結果までの時間が短くなります。セッションが開始されると、QPUはQESEMジョブ専用に予約されます。使用量の計算には、QPU実行に費やした時間と関連する古典的な計算（QESEMとIBMによって実行される）の両方が含まれます。QESEM Qiskit Functionはセッションの作成とクローズを自動的に処理します。QPUへの無制限アクセスを持つユーザー（例：オンプレミスのセットアップ）には、より高速なQESEM実行のためにセッションモードの使用が推奨されます。
    -   **バッチモード**: バッチモードでは、古典的な計算中にQPUが解放されるため、QPUの使用量が少なくなります。バッチジョブは通常、より長い期間にわたるため、ハードウェアドリフトのリスクが高まります。QESEMはドリフトを検出して補償する手段を組み込んでおり、長期間にわたって信頼性を維持します。

> **Note:** バリア操作は通常、量子回路における2量子ビットゲートのレイヤーを指定するために使用されます。レベル0では、QESEMはバリアで指定されたレイヤーを保持します。レベル1では、バリアで指定されたレイヤーはQPU時間を最小化する際の1つのトランスパイレーション代替案として考慮されます。
### 出力
Circuit functionの出力は[PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)であり、2つのフィールドを含みます：

- 1つの[PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult)オブジェクト。`PrimitiveResult`から直接インデックスできます。

- ジョブレベルのメタデータ。

各`PubResult`には`data`フィールドと`metadata`フィールドが含まれます。

- `data`フィールドには、少なくとも期待値の配列（`PubResult.data.evs`）と標準誤差の配列（`PubResult.data.stds`）が含まれます。使用されたオプションに応じて、追加のデータが含まれる場合もあります。

- `metadata`フィールドにはPUBレベルのメタデータ（`PubResult.metadata`）が含まれます。

以下のコードスニペットは、QPU時間の見積もりを取得する方法を示しています（`estimate_time_only`が設定されている場合）：

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

以下のコードスニペットは、緩和結果（`estimate_time_only`が設定されていない場合）と実行メトリクスを取得する方法を示しています。これらには、さまざまなパラメータがQESEM実行にどのように影響するかをより深く理解するための重要なデータが含まれています。研究に基づいて論文を書く際にも関連する可能性があります。

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## エラーメッセージの取得
ワークロードのステータスがERRORの場合、job.result()を使用して以下のようにエラーメッセージを取得してください：

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>